<a href="https://colab.research.google.com/github/Ashleyyy678/CS480/blob/main/Ashley_Nwadike_CS_449_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# CS617 Assignment 3 - Spring 2026
# Boston Shooting Incidents Analysis
# Name: [Your Name]

import pandas as pd
import urllib.request
import urllib.parse
import json
from datetime import datetime

print("="*60)
print("BOSTON SHOOTING INCIDENTS - DATA STORY")
print("="*60)

# 1. FETCH DATA FROM BOSTON PD API
sql = '''
SELECT "DISTRICT",
       "OCCURRED_ON_DATE",
       "SHOOTING",
       "OFFENSE_DESCRIPTION"
FROM "b973d8cb-eeb2-4e7e-99da-c92938efc9c0"
WHERE CAST("SHOOTING" AS INT)=1
'''

bpd_logs = "https://data.boston.gov/api/3/action/datastore_search_sql"
url = bpd_logs + "?sql=" + urllib.parse.quote(sql)

print("\n📡 Fetching shooting data from Boston PD...")
with urllib.request.urlopen(url) as response:
    data = json.loads(response.read().decode())

shootings = data['result']['records']
print(f"✅ Retrieved {len(shootings)} shooting incidents")

# 2. CREATE DATAFRAME
df = pd.DataFrame(shootings)
df['OCCURRED_ON_DATE'] = pd.to_datetime(df['OCCURRED_ON_DATE'])
df = df.dropna(subset=['DISTRICT'])

# 3. EXTRACT TIME FEATURES
df['YEAR'] = df['OCCURRED_ON_DATE'].dt.year
df['MONTH'] = df['OCCURRED_ON_DATE'].dt.month
df['MONTH_NAME'] = df['OCCURRED_ON_DATE'].dt.strftime('%b')
df['YEAR_MONTH'] = df['OCCURRED_ON_DATE'].dt.strftime('%Y-%m')

print(f"\n📊 Data Summary:")
print(f"   Total incidents: {len(df)}")
print(f"   Date range: {df['OCCURRED_ON_DATE'].min().date()} to {df['OCCURRED_ON_DATE'].max().date()}")
print(f"   Districts involved: {df['DISTRICT'].nunique()}")

# 4. CREATE CHART 1 DATA: Shootings by District
print("\n📊 CHART 1: Shootings by District")
district_counts = df['DISTRICT'].value_counts().reset_index()
district_counts.columns = ['district', 'count']
district_counts = district_counts.sort_values('count', ascending=False).head(15)

print("\nTop 10 Districts:")
for idx, row in district_counts.head(10).iterrows():
    print(f"   {row['district']}: {row['count']} incidents")

# 5. CREATE CHART 2 DATA: Monthly Trends
print("\n📈 CHART 2: Monthly Trends")

# Overall monthly trend
monthly_trend = df.groupby('YEAR_MONTH').size().reset_index()
monthly_trend.columns = ['month', 'count']
monthly_trend = monthly_trend.sort_values('month')

print(f"\nOverall monthly trend (sample):")
print(monthly_trend.head(6).to_string(index=False))

# District-specific trends (for interactivity)
district_trends = {}
for district in district_counts['district'].head(8):  # Top 8 districts
    district_df = df[df['DISTRICT'] == district]
    trend = district_df.groupby('YEAR_MONTH').size().reset_index()
    trend.columns = ['month', 'count']
    trend = trend.sort_values('month')
    district_trends[district] = trend.to_dict('records')

# 6. CREATE JSON FOR PLOTLY
output_data = {
    "story_title": "Shooting Incidents in Boston: Hotspots and Seasonal Patterns",
    "summary": {
        "total_incidents": int(len(df)),
        "date_range": {
            "start": df['OCCURRED_ON_DATE'].min().strftime('%Y-%m-%d'),
            "end": df['OCCURRED_ON_DATE'].max().strftime('%Y-%m-%d')
        },
        "top_district": district_counts.iloc[0]['district'],
        "top_district_count": int(district_counts.iloc[0]['count'])
    },
    "chart1_districts": district_counts.to_dict('records'),
    "chart2_trend": monthly_trend.to_dict('records'),
    "district_trends": district_trends
}

# 7. SAVE JSON FILE
filename = 'boston_shooting_story.json'
with open(filename, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"\n✅ Data saved to '{filename}'")

# 8. DOWNLOAD FILE
from google.colab import files
files.download(filename)

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("""
1. Download the JSON file
2. Create an HTML file with two Plotly charts:
   - Chart 1: Bar chart of shootings by district
   - Chart 2: Monthly trend line (updates when district clicked)
3. Host on GitHub Pages
4. Submit link: https://[username].github.io/[repo]/asst3.html
""")

BOSTON SHOOTING INCIDENTS - DATA STORY

📡 Fetching shooting data from Boston PD...
✅ Retrieved 1721 shooting incidents

📊 Data Summary:
   Total incidents: 1720
   Date range: 2023-01-01 to 2026-03-03
   Districts involved: 13

📊 CHART 1: Shootings by District

Top 10 Districts:
   B3: 419 incidents
   B2: 413 incidents
   C11: 333 incidents
   E13: 117 incidents
   E18: 110 incidents
   D4: 85 incidents
   C6: 60 incidents
   E5: 56 incidents
   D14: 40 incidents
   A1: 40 incidents

📈 CHART 2: Monthly Trends

Overall monthly trend (sample):
  month  count
2023-01     52
2023-02     36
2023-03     35
2023-04     42
2023-05     50
2023-06     63

✅ Data saved to 'boston_shooting_story.json'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


NEXT STEPS:

1. Download the JSON file
2. Create an HTML file with two Plotly charts:
   - Chart 1: Bar chart of shootings by district
   - Chart 2: Monthly trend line (updates when district clicked)
3. Host on GitHub Pages
4. Submit link: https://[username].github.io/[repo]/asst3.html

